In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import ipywidgets as widgets

In [11]:

# Cargar los datos
datos = pd.read_csv("datos_ventas_multiples_productos.csv")

# Convertimos la columna de mes a un valor numérico (mes) si es una fecha completa
datos["mes"] = pd.to_datetime(datos["mes"]).dt.month

# Lista de productos únicos
productos = datos["producto"].unique()

In [12]:

def plot_predictions(mes_pred):
    plt.figure(figsize=(15, len(productos) * 5))

    for i, producto in enumerate(productos):
        # Filtrar los datos por producto
        datos_producto = datos[datos["producto"] == producto]
        
        # Asegurarse de que hay datos suficientes para hacer predicciones
        if datos_producto.empty:
            print(f"No hay datos disponibles para el producto {producto}.")
            continue
        
        # Definir variables predictoras y objetivos
        X = datos_producto[["mes", "precio_venta"]]
        y_unidades = datos_producto["unidades_vendidas"]
        y_precio = datos_producto["precio_venta"]
        y_compra = datos_producto["precio_compra"]
        
        # Dividir los datos en conjuntos de entrenamiento y prueba
        X_train, X_test, y_unidades_train, y_unidades_test = train_test_split(X, y_unidades, test_size=0.2, random_state=42)
        _, _, y_precio_train, y_precio_test = train_test_split(X, y_precio, test_size=0.2, random_state=42)
        _, _, y_compra_train, y_compra_test = train_test_split(X, y_compra, test_size=0.2, random_state=42)
        
        # Entrenar modelos: Bosque Aleatorio para Unidades, Regresión Lineal para Precios
        modelo_unidades = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_unidades_train)
        modelo_precio = LinearRegression().fit(X_train, y_precio_train)
        modelo_compra = LinearRegression().fit(X_train, y_compra_train)
        
        # Definir el nuevo mes para la predicción
        nuevo_mes = pd.DataFrame({"mes": [mes_pred], "precio_venta": [y_precio.mean()]})
        
        # Predicciones para el mes elegido
        unidades_pred = modelo_unidades.predict(nuevo_mes)[0]
        precio_venta_pred = modelo_precio.predict(nuevo_mes)[0]
        precio_compra_pred = modelo_compra.predict(nuevo_mes)[0]
        
        # Cálculo de error cuadrático medio
        mse_unidades = mean_squared_error(y_unidades_test, modelo_unidades.predict(X_test))
        mse_precio = mean_squared_error(y_precio_test, modelo_precio.predict(X_test))
        mse_compra = mean_squared_error(y_compra_test, modelo_compra.predict(X_test))
        
        # Mostrar los errores
        print(f"\nProducto {producto}:")
        print(f"  - Error cuadrático medio para Unidades (Bosque Aleatorio): {mse_unidades:.2f}")
        print(f"  - Error cuadrático medio para Precio de Venta: {mse_precio:.2f}")
        print(f"  - Error cuadrático medio para Precio de Compra: {mse_compra:.2f}")
        
        # Gráficos para el producto actual
        plt.subplot(len(productos), 3, i * 3 + 1)
        plt.plot(datos_producto["mes"], datos_producto["unidades_vendidas"], label="Unidades históricas", marker='o')
        plt.scatter(mes_pred, unidades_pred, color="red", label="Predicción elegida")
        plt.text(mes_pred, unidades_pred, f"{unidades_pred:.0f}", color="red", ha='center', va='bottom')
        plt.xlabel("Mes")
        plt.ylabel("Unidades vendidas")
        plt.title(f"Predicción de Unidades - Producto {producto}")
        plt.legend()
        
        plt.subplot(len(productos), 3, i * 3 + 2)
        plt.plot(datos_producto["mes"], datos_producto["precio_venta"], label="Precio venta histórico", marker='o')
        plt.scatter(mes_pred, precio_venta_pred, color="red", label="Predicción elegida")
        plt.text(mes_pred, precio_venta_pred, f"{precio_venta_pred:.2f}", color="red", ha='center', va='bottom')
        plt.xlabel("Mes")
        plt.ylabel("Precio de venta")
        plt.title(f"Predicción de Precio de Venta - Producto {producto}")
        plt.legend()
        
        plt.subplot(len(productos), 3, i * 3 + 3)
        plt.plot(datos_producto["mes"], datos_producto["precio_compra"], label="Precio compra histórico", marker='o')
        plt.scatter(mes_pred, precio_compra_pred, color="red", label="Predicción elegida")
        plt.text(mes_pred, precio_compra_pred, f"{precio_compra_pred:.2f}", color="red", ha='center', va='bottom')
        plt.xlabel("Mes")
        plt.ylabel("Precio de compra óptimo")
        plt.title(f"Predicción de Precio de Compra - Producto {producto}")
        plt.legend()

    # Ajustar el diseño y mostrar todos los gráficos
    plt.tight_layout()
    plt.show()

# Crear un slider para seleccionar el mes de predicción
mes_slider = widgets.IntSlider(value=15, min=1, max=50, description='Mes a predecir:')
widgets.interactive(plot_predictions, mes_pred=mes_slider)

interactive(children=(IntSlider(value=15, description='Mes a predecir:', max=50, min=1), Output()), _dom_class…